# Milestone 6 — Residual model gaps and adaptation hypothesis

Milestone 5 improved the frozen Ministral-3B result from the reported minimal
baseline of 1/24 to **7/24**, while execution success reached **17/24**. That is
meaningful evidence that context engineering helps, but it does not yet prove
that fine-tuning is the correct next intervention.

This notebook asks the PRD's next question:

> After the selected context and deterministic improvements, are the remaining
> failures systematic, repetitive, and learnable enough to justify an SFT/LoRA
> experiment?

The answer will be based on detailed `working_dev` evidence only. Frozen results
contribute aggregate measurements and artifact hashes; their questions, SQL,
outputs, and per-question outcomes remain sealed.

No model request and no training operation occurs in this notebook.


## Decision standard

Fine-tuning is justified as an experiment only when all of the following are
reasonably supported:

1. **The required information was available.** The selected runtime context
   already supplied the relevant tables, relationships, meanings, and values.
2. **The failure pattern repeats.** More than one question exhibits the same
   behavioral weakness.
3. **Deterministic/context work has reached diminishing returns.** Adding more
   prose or a generic repair call did not solve the pattern reliably.
4. **The behavior is teachable.** Verified, leakage-safe training examples can
   demonstrate the desired mapping or construction behavior.
5. **The fact belongs in behavior rather than weights.** Dynamic schema and
   database values must remain runtime context.

This is a go/no-go decision for **Milestone 7 training-data design**, not a
promise that adaptation will outperform the selected system.


## Leakage boundary and required local artifacts

Detailed analysis is limited to the 12 working-development questions. The
notebook loads these ignored local artifacts:

- `gpt_5_2_minimal_working_dev_v1.json`
- `ministral_3b_minimal_working_dev_v1.json`
- `ministral_3b_compact_guardrails_strict_unwrap_working_dev_v1.json`

It does not parse a frozen run. If the official Milestone 5 V2 artifact is
present, only its raw-file SHA-256 is checked.


In [1]:
from collections import Counter
from hashlib import sha256
from pathlib import Path
import json
import sqlite3

from domain_adapt.evaluation import compare_sql_results
from domain_adapt.finch import make_reference_compatible


DATA_DIR = Path("../data").resolve()
DATASET_PATH = DATA_DIR / "finch_dataset.json"
DB_PATH = DATA_DIR / "financial.sqlite"
MANIFEST_PATH = Path("../configs/eval_manifest_v1.json").resolve()
CONTEXT_CONFIG_PATH = Path(
    "../configs/slm_context_engineered_v1.json"
).resolve()

FRONTIER_WORKING_PATH = (
    DATA_DIR / "runs" / "gpt_5_2_minimal_working_dev_v1.json"
)
MINIMAL_SLM_WORKING_PATH = (
    DATA_DIR / "runs" / "ministral_3b_minimal_working_dev_v1.json"
)
SELECTED_WORKING_PATH = (
    DATA_DIR / "runs" /
    "ministral_3b_compact_guardrails_strict_unwrap_working_dev_v1.json"
)
OFFICIAL_FROZEN_V2_PATH = (
    DATA_DIR / "runs" / "sealed" /
    "ministral_3b_context_engineered_frozen_eval_v2.json"
)

required_paths = (
    DATASET_PATH,
    DB_PATH,
    MANIFEST_PATH,
    CONTEXT_CONFIG_PATH,
    FRONTIER_WORKING_PATH,
    MINIMAL_SLM_WORKING_PATH,
    SELECTED_WORKING_PATH,
)
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        f"Missing Milestone 6 prerequisites: {missing_paths}"
    )

records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
context_config = json.loads(
    CONTEXT_CONFIG_PATH.read_text(encoding="utf-8")
)
frontier_run = json.loads(
    FRONTIER_WORKING_PATH.read_text(encoding="utf-8")
)
minimal_slm_run = json.loads(
    MINIMAL_SLM_WORKING_PATH.read_text(encoding="utf-8")
)
selected_run = json.loads(
    SELECTED_WORKING_PATH.read_text(encoding="utf-8")
)
conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
assert working_ids.isdisjoint(frozen_ids)

EXPECTED_CONTEXT_SHA256 = (
    "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987"
)
EXPECTED_INSTRUCTIONS_SHA256 = (
    "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509"
)
EXPECTED_FROZEN_V2_SHA256 = (
    "f904ccffa106648265537f28b39a72c92f484950c957355ef183bda755adbd67"
)

assert context_config["schema_context_sha256"] == EXPECTED_CONTEXT_SHA256
assert context_config["instructions_sha256"] == EXPECTED_INSTRUCTIONS_SHA256
assert sha256(
    context_config["schema_context"].encode("utf-8")
).hexdigest() == EXPECTED_CONTEXT_SHA256
assert sha256(
    context_config["instructions"].encode("utf-8")
).hexdigest() == EXPECTED_INSTRUCTIONS_SHA256

if OFFICIAL_FROZEN_V2_PATH.exists():
    # Hash bytes only. Do not parse or inspect frozen contents.
    assert sha256(
        OFFICIAL_FROZEN_V2_PATH.read_bytes()
    ).hexdigest() == EXPECTED_FROZEN_V2_SHA256
    print("Official frozen V2 artifact hash validated without parsing it.")
else:
    print("Official frozen V2 artifact not present; recorded aggregate evidence remains usable.")

print("Working IDs:", len(working_ids))
print("Frozen records inspected:", 0)
print("No model request made.")


Official frozen V2 artifact hash validated without parsing it.
Working IDs: 12
Frozen records inspected: 0
No model request made.


## Validate comparable working runs

All three runs must cover exactly the same 12 working questions. The selected
run must carry the frozen Milestone 5 context and instruction hashes. This
prevents an accidental comparison across different prompts or data slices.


In [2]:
def results_by_id(run: dict) -> dict[int, dict]:
    results = run["results"]
    by_id = {result["question_id"]: result for result in results}
    assert len(by_id) == len(results)
    assert set(by_id) == working_ids
    assert not (set(by_id) & frozen_ids)
    return by_id


frontier_by_id = results_by_id(frontier_run)
minimal_by_id = results_by_id(minimal_slm_run)
selected_by_id = results_by_id(selected_run)

assert selected_run["context_sha256"] == EXPECTED_CONTEXT_SHA256
assert selected_run["instructions_sha256"] == EXPECTED_INSTRUCTIONS_SHA256
assert selected_run["attempts_per_question"] == 1
assert selected_run["candidate_sql_repair"] is False


def is_executable(result: dict) -> bool:
    reason = result.get("reason") or ""
    return (
        result.get("api_error") is None
        and not reason.startswith("candidate execution failed")
        and not reason.startswith("model request failed")
    )


def run_summary(by_id: dict[int, dict]) -> dict:
    results = list(by_id.values())
    return {
        "passed": sum(result["passed"] for result in results),
        "total": len(results),
        "executable": sum(is_executable(result) for result in results),
    }


working_summaries = {
    "Frontier baseline": run_summary(frontier_by_id),
    "Minimal Ministral": run_summary(minimal_by_id),
    "Selected Milestone 5": run_summary(selected_by_id),
}

for name, summary in working_summaries.items():
    print(
        f"{name:<24} "
        f"accuracy={summary['passed']}/{summary['total']} "
        f"executable={summary['executable']}/{summary['total']}"
    )


Frontier baseline        accuracy=4/12 executable=12/12
Minimal Ministral        accuracy=1/12 executable=5/12
Selected Milestone 5     accuracy=4/12 executable=8/12


## Measure question-level transitions

Aggregate uplift does not reveal mechanism. The transition table separates:

- newly corrected questions;
- retained successes;
- execution recovery without semantic correctness;
- residual failures; and
- regressions, if any.


In [3]:
transitions = []

for question_id in sorted(working_ids):
    minimal = minimal_by_id[question_id]
    selected = selected_by_id[question_id]

    if not minimal["passed"] and selected["passed"]:
        transition = "newly_correct"
    elif minimal["passed"] and selected["passed"]:
        transition = "retained_correct"
    elif minimal["passed"] and not selected["passed"]:
        transition = "regressed"
    elif not is_executable(minimal) and is_executable(selected):
        transition = "execution_recovered_but_incorrect"
    else:
        transition = "still_incorrect"

    transitions.append({
        "question_id": question_id,
        "difficulty": selected["difficulty"],
        "minimal_passed": minimal["passed"],
        "selected_passed": selected["passed"],
        "minimal_executable": is_executable(minimal),
        "selected_executable": is_executable(selected),
        "transition": transition,
    })

transition_counts = Counter(
    row["transition"] for row in transitions
)

print("Transition counts:", dict(transition_counts))
print()
print(f"{'ID':>3} {'Difficulty':<8} {'Minimal':<10} {'Selected':<10} Transition")
print("-" * 68)
for row in transitions:
    print(
        f"{row['question_id']:>3} "
        f"{row['difficulty']:<8} "
        f"{'PASS' if row['minimal_passed'] else 'FAIL':<10} "
        f"{'PASS' if row['selected_passed'] else 'FAIL':<10} "
        f"{row['transition']}"
    )

assert transition_counts["regressed"] == 0
assert transition_counts["newly_correct"] == 3
assert transition_counts["retained_correct"] == 1


Transition counts: {'newly_correct': 3, 'still_incorrect': 5, 'execution_recovered_but_incorrect': 3, 'retained_correct': 1}

 ID Difficulty Minimal    Selected   Transition
--------------------------------------------------------------------
  5 medium   FAIL       PASS       newly_correct
 11 medium   FAIL       FAIL       still_incorrect
 14 easy     FAIL       FAIL       execution_recovered_but_incorrect
 16 easy     PASS       PASS       retained_correct
 45 easy     FAIL       FAIL       still_incorrect
 49 medium   FAIL       FAIL       execution_recovered_but_incorrect
 61 hard     FAIL       FAIL       execution_recovered_but_incorrect
 69 easy     FAIL       PASS       newly_correct
 81 hard     FAIL       FAIL       still_incorrect
 89 easy     FAIL       FAIL       still_incorrect
 99 easy     FAIL       PASS       newly_correct
105 medium   FAIL       FAIL       still_incorrect


### What Milestone 5 actually fixed

The selected system retained the one minimal-model success and corrected three
additional questions. It also made three previously non-executable outputs
executable without making them correct. That distinction matters:

- context and guardrails clearly improved model behavior;
- syntactic/execution recovery was easier than semantic recovery;
- no previously correct working answer regressed;
- eight questions remain incorrect and require root-cause analysis.


## Inspect residual working failures

The next cell displays only working-development evidence: question, compatible
reference SQL, selected generated SQL, evaluator reason, and small result
previews. This is diagnostic evidence, not training material. None of these
questions may be copied, paraphrased, or used to generate SFT examples.


In [4]:
record_by_id = {
    row["question_id"]: row
    for row in records
    if row["db_name"] == "bird" and row["db_id"] == "financial"
}


def query_preview(sql: str | None, limit: int = 5) -> dict:
    if sql is None:
        return {"error": "No candidate SQL"}
    try:
        cursor = conn.execute(sql)
        return {
            "columns": [item[0] for item in cursor.description or []],
            "rows": cursor.fetchmany(limit),
        }
    except Exception as error:
        return {"error": f"{type(error).__name__}: {error}"}


residual_ids = {
    question_id
    for question_id, result in selected_by_id.items()
    if not result["passed"]
}
EXPECTED_RESIDUAL_IDS = {11, 14, 45, 49, 61, 81, 89, 105}
assert residual_ids == EXPECTED_RESIDUAL_IDS

residual_evidence = {}

for question_id in sorted(residual_ids):
    example = record_by_id[question_id]
    result = selected_by_id[question_id]
    reference_sql, rules = make_reference_compatible(example["SQL"])
    comparison = compare_sql_results(
        conn=conn,
        reference_sql=reference_sql,
        candidate_sql=result["generated_sql"],
    )
    assert comparison.equivalent is False
    assert comparison.reason == result["reason"]

    residual_evidence[question_id] = {
        "question": example["question"],
        "reference_sql": reference_sql,
        "generated_sql": result["generated_sql"],
        "reason": result["reason"],
        "reference_rules": rules,
        "reference_preview": query_preview(reference_sql),
        "candidate_preview": query_preview(result["generated_sql"]),
    }

    evidence = residual_evidence[question_id]
    print("=" * 88)
    print("Question ID:", question_id)
    print("Question:", evidence["question"])
    print("Reference SQL:\n", evidence["reference_sql"])
    print("Selected SQL:\n", evidence["generated_sql"])
    print("Evaluator reason:", evidence["reason"])
    print("Reference preview:", evidence["reference_preview"])
    print("Candidate preview:", evidence["candidate_preview"])


Question ID: 11
Question: Among the accounts who have loan validity more than 12 months, list out the accounts that have the highest approved amount and have account opening date in 1993.
Reference SQL:
 SELECT t1.account_id FROM loan AS t1 INNER JOIN account AS t2 ON t1.account_id = t2.account_id WHERE STRFTIME('%Y', t2.date) = '1993' AND t1.duration > 12 ORDER BY t1.amount DESC LIMIT 1
Selected SQL:
 SELECT account_id
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE loan.duration > 12
AND account.date = '1993-01-01'
ORDER BY loan.amount DESC
LIMIT 1;
Evaluator reason: candidate execution failed: ambiguous column name: account_id
Reference preview: {'columns': ['account_id'], 'rows': [(10451,)]}
Candidate preview: {'error': 'OperationalError: ambiguous column name: account_id'}
Question ID: 14
Question: State different accounts who have account opening date before 1997 and own an amount of money greater than 3000USD
Reference SQL:
 SELECT DISTINCT t2.account_id FRO

## Evidence-backed residual taxonomy

A failure can have several contributing causes, but one primary behavioral gap
is selected for counting. `adaptation_candidate` means the behavior is plausibly
learnable from disjoint verified examples; it does not mean the working question
itself may enter training data.


In [6]:
residual_labels = {
    11: {
        "primary_gap": "identifier_and_filter_discipline",
        "cluster": "schema_constrained_sql_construction",
        "contributing_gaps": ["temporal_filter_reasoning"],
        "context_was_available": True,
        "adaptation_candidate": True,
        "confidence": "high",
        "evidence": (
            "The selected query remained non-executable because account_id was "
            "ambiguous across joined tables. The prompt explicitly required "
            "qualified columns, so the failure is behavioral rather than missing context."
        ),
    },
    14: {
        "primary_gap": "schema_linking_and_projection",
        "cluster": "schema_constrained_sql_construction",
        "contributing_gaps": ["column_selection", "requested_output_shape"],
        "context_was_available": True,
        "adaptation_candidate": True,
        "confidence": "high",
        "evidence": (
            "The selected query used the wrong financial table/measure and "
            "returned an extra column. Table relationships, amount meanings, "
            "and the return-only-requested-columns rule were all supplied."
        ),
    },
    45: {
        "primary_gap": "opaque_column_and_projection_mapping",
        "cluster": "schema_constrained_sql_construction",
        "contributing_gaps": ["ranking_logic", "requested_output_shape"],
        "context_was_available": True,
        "adaptation_candidate": True,
        "confidence": "medium",
        "evidence": (
            "The query still returned the wrong ranked district result despite "
            "explicit A12/A13 year meanings and output-column instructions."
        ),
    },
    49: {
        "primary_gap": "multi_value_business_predicate",
        "cluster": "domain_language_to_sql",
        "contributing_gaps": ["join_and_filter_construction"],
        "context_was_available": True,
        "adaptation_candidate": True,
        "confidence": "high",
        "evidence": (
            "The context explicitly defines running loans as both C and D and "
            "supplies the account-to-loan relationship, yet the selected result "
            "did not match the required count."
        ),
    },
    61: {
        "primary_gap": "domain_role_semantics",
        "cluster": "domain_language_to_sql",
        "contributing_gaps": ["requested_output_mapping", "benchmark_ambiguity"],
        "context_was_available": True,
        "adaptation_candidate": "conditional",
        "confidence": "medium",
        "evidence": (
            "The benchmark maps loan eligibility to disposition ownership, but "
            "the model returned account-frequency semantics. Because the wording "
            "is ambiguous, only unambiguous disjoint examples should teach this behavior."
        ),
    },
    81: {
        "primary_gap": "multi_table_aggregation_and_arithmetic",
        "cluster": "multi_step_query_reasoning",
        "contributing_gaps": ["join_path", "percentage_calculation"],
        "context_was_available": True,
        "adaptation_candidate": True,
        "confidence": "high",
        "evidence": (
            "The task requires the loan-account-disposition-client path plus "
            "yearly conditional aggregation and percentage growth. The supplied "
            "relationships and value mappings were not converted into valid SQL."
        ),
    },
    89: {
        "primary_gap": "multi_table_join_path",
        "cluster": "schema_constrained_sql_construction",
        "contributing_gaps": ["nonexistent_column", "aggregation_semantics"],
        "context_was_available": True,
        "adaptation_candidate": True,
        "confidence": "high",
        "evidence": (
            "The selected SQL attempted a direct account-to-client relationship "
            "through a nonexistent column instead of using disp, even though the "
            "foreign-key path was explicitly provided."
        ),
    },
    105: {
        "primary_gap": "multi_table_join_and_projection",
        "cluster": "schema_constrained_sql_construction",
        "contributing_gaps": ["identifier_qualification", "domain_role_semantics"],
        "context_was_available": True,
        "adaptation_candidate": True,
        "confidence": "high",
        "evidence": (
            "The task requires the owner role and exact client/district output. "
            "The selected query used invalid or ambiguous relationships and did "
            "not preserve the requested projection."
        ),
    },
}

assert set(residual_labels) == residual_ids
for question_id, label in residual_labels.items():
    assert label["primary_gap"]
    assert label["cluster"]
    assert label["context_was_available"] is True
    assert label["adaptation_candidate"] in {True, False, "conditional"}
    assert label["confidence"] in {"low", "medium", "high"}
    assert len(label["evidence"].split()) >= 15

print("Every residual failure has an evidence-backed label:", len(residual_labels))


Every residual failure has an evidence-backed label: 8


In [7]:
cluster_counts = Counter(
    label["cluster"] for label in residual_labels.values()
)
primary_gap_counts = Counter(
    label["primary_gap"] for label in residual_labels.values()
)
adaptation_counts = Counter(
    str(label["adaptation_candidate"])
    for label in residual_labels.values()
)

print("Residual clusters:", dict(cluster_counts))
print("Primary gaps:", dict(primary_gap_counts))
print("Adaptation candidacy:", dict(adaptation_counts))

assert cluster_counts == {
    "schema_constrained_sql_construction": 5,
    "domain_language_to_sql": 2,
    "multi_step_query_reasoning": 1,
}
assert adaptation_counts == {
    "True": 7,
    "conditional": 1,
}


Residual clusters: {'schema_constrained_sql_construction': 5, 'domain_language_to_sql': 2, 'multi_step_query_reasoning': 1}
Primary gaps: {'identifier_and_filter_discipline': 1, 'schema_linking_and_projection': 1, 'opaque_column_and_projection_mapping': 1, 'multi_value_business_predicate': 1, 'domain_role_semantics': 1, 'multi_table_aggregation_and_arithmetic': 1, 'multi_table_join_path': 1, 'multi_table_join_and_projection': 1}
Adaptation candidacy: {'True': 7, 'conditional': 1}


## What belongs in runtime context versus model behavior

The experiment should not train database facts into the weights merely to save
prompt tokens.

| Keep at runtime | Plausible adaptation target |
|---|---|
| table and column names | obey supplied table-column ownership |
| foreign-key relationships | construct joins only through supplied paths |
| opaque-column meanings such as A11/A13 | apply supplied meanings to the question |
| exact categorical codes | use supplied literals without inventing English values |
| date storage formats | construct valid year predicates from supplied formats |
| database-specific business glossary | follow exact requested projection |
| changing schema and values | avoid unrequested filters and nonexistent columns |
| retrieved evidence | perform recurring aggregation/arithmetic patterns |

This boundary preserves adaptability: when another database is introduced, its
facts can change without retraining, while the model retains better disciplined
SQL-construction behavior.


## Why another generic repair loop is not the next experiment

Milestone 5 already tested one execution-error repair pass. Executability rose
from 8/12 to 10/12, but accuracy remained 4/12 and the repair added 6,495 tokens.
Repair therefore treated symptoms without recovering correct answers.

The residual taxonomy also includes executable-but-wrong queries, which SQLite
cannot identify without an oracle. Adding more execution retries would increase
cost while leaving semantic failures invisible. This supports studying learned
construction behavior rather than building an open-ended agent loop.


## Aggregate frozen evidence

Frozen details remain sealed. Only previously recorded aggregates are used:

| System | Accuracy | Executable | Prompt tokens | Status |
|---|---:|---:|---:|---|
| GPT-5.2 minimal | 11/24 (45.8%) | 24/24 | 7,077 | complete |
| Ministral minimal | 1/24 (4.2%) | approximately 10/24 | 6,919 | one API failure; lower-bound result |
| Ministral + selected Milestone 5 system | **7/24 (29.2%)** | **17/24 (70.8%)** | 32,739 | complete V2 |

The selected system recovered approximately 64% of frontier aggregate task
accuracy, but it used about 4.7 times the minimal Ministral prompt tokens and
still failed every hard question. The single missing minimal-baseline response
must be completed before publishing the final benchmark table, but it cannot
erase the large remaining frontier gap.


In [10]:
frozen_aggregate_evidence = {
    "frontier_minimal": {
        "passed": 11,
        "total": 24,
        "executable": 24,
        "prompt_tokens": 7077,
        "api_errors": 0,
    },
    "ministral_minimal_reported": {
        "passed": 1,
        "total": 24,
        "prompt_tokens": 6919,
        "api_errors": 1,
        "status": "lower_bound_pending_one_infrastructure_completion",
    },
    "ministral_context_engineered": {
        "passed": 7,
        "total": 24,
        "executable": 17,
        "prompt_tokens": 32739,
        "api_errors": 0,
        "sealed_run_sha256": EXPECTED_FROZEN_V2_SHA256,
    },
}

context_accuracy = (
    frozen_aggregate_evidence["ministral_context_engineered"]["passed"] / 24
)
frontier_accuracy = (
    frozen_aggregate_evidence["frontier_minimal"]["passed"] / 24
)
recovered_frontier_fraction = context_accuracy / frontier_accuracy
prompt_multiplier = (
    frozen_aggregate_evidence["ministral_context_engineered"]["prompt_tokens"]
    / frozen_aggregate_evidence["ministral_minimal_reported"]["prompt_tokens"]
)

print(f"Fraction of frontier aggregate accuracy recovered: {recovered_frontier_fraction:.1%}")
print(f"Prompt-token multiplier versus minimal Ministral: {prompt_multiplier:.2f}x")


Fraction of frontier aggregate accuracy recovered: 63.6%
Prompt-token multiplier versus minimal Ministral: 4.73x


## Adaptation hypothesis

**Decision: proceed to Milestone 7 training-data design, with a targeted and
falsifiable SFT/LoRA experiment.**

The evidence does not justify training schema facts into the model. It supports
a narrower behavioral hypothesis:

> Given complete and correct runtime schema/domain context, Ministral-3B
> systematically struggles to translate that context into schema-constrained SQL—
> especially valid multi-table join paths, exact requested projections, qualified
> identifiers, and multi-step aggregation. SFT/LoRA on leakage-safe, executable,
> disjoint examples of these behaviors should improve execution and result
> accuracy while retaining runtime grounding for database-specific facts.

Question 61's ambiguous eligibility language is conditional evidence and should
not drive training unless unambiguous source examples define the same mapping.

If Ministral-3B is not available for fine-tuning and a different SLM is selected,
that model must first receive an unadapted baseline under the identical Milestone 5
runtime contract. Otherwise, a post-training difference could not be attributed to
adaptation rather than to changing the underlying model.


In [11]:
adaptation_hypothesis = {
    "version": 1,
    "decision": "proceed_to_training_data_design",
    "model": "Ministral-3B or another Azure-supported fine-tuning candidate",
    "model_change_policy": (
        "If the fine-tuning model differs from Ministral-3B, establish its "
        "unadapted baseline under the identical Milestone 5 runtime contract "
        "before training."
    ),
    "fixed_runtime_system": {
        "context_sha256": EXPECTED_CONTEXT_SHA256,
        "instructions_sha256": EXPECTED_INSTRUCTIONS_SHA256,
        "output_normalization": "strict single surrounding SQL-fence unwrap",
        "candidate_sql_repair": False,
    },
    "hypothesis": (
        "Given complete and correct runtime schema/domain context, the SLM "
        "systematically struggles with schema-constrained SQL construction, "
        "including multi-table join paths, exact projection, identifier "
        "qualification, and multi-step aggregation. SFT/LoRA on verified, "
        "leakage-safe, disjoint examples of those behaviors should improve "
        "execution and result accuracy without moving database-specific facts "
        "into model weights."
    ),
    "training_targets": [
        "join only through supplied foreign-key paths",
        "use columns only from their owning table or alias",
        "return exactly the requested columns in the requested order",
        "qualify identifiers in multi-table queries",
        "avoid unrequested filters and assumptions",
        "construct recurring conditional aggregation and percentage patterns",
    ],
    "runtime_only": [
        "table and column names",
        "foreign-key relationships",
        "database-specific column meanings",
        "categorical literals and business definitions",
        "date formats",
        "changing schema and retrieved evidence",
    ],
    "excluded_from_training": {
        "working_question_ids": sorted(working_ids),
        "frozen_question_ids": sorted(frozen_ids),
        "working_questions_or_sql_as_synthetic_seeds": True,
        "frozen_questions_or_sql_as_synthetic_seeds": True,
    },
    "pre_registered_success_criteria": {
        "primary_frozen_task_success": "at least 10/24",
        "minimum_frozen_execution_success": "at least 20/24",
        "working_pass_regressions": 0,
        "api_errors": 0,
        "comparison_system": "identical Milestone 5 runtime contract",
        "attempts_per_question": 1,
        "candidate_sql_repair": False,
    },
    "failure_conditions": [
        "frozen task success remains below 10/24",
        "frozen execution success remains below 20/24",
        "working examples that previously passed regress",
        "gains require leakage, retries, repair, or a changed runtime contract",
        "deployment economics outweigh measured quality improvement",
    ],
}

assert adaptation_hypothesis["decision"] == "proceed_to_training_data_design"
assert adaptation_hypothesis["fixed_runtime_system"]["context_sha256"] == (
    EXPECTED_CONTEXT_SHA256
)
assert adaptation_hypothesis["fixed_runtime_system"]["instructions_sha256"] == (
    EXPECTED_INSTRUCTIONS_SHA256
)
assert set(adaptation_hypothesis["excluded_from_training"]["working_question_ids"]) == (
    working_ids
)
assert set(adaptation_hypothesis["excluded_from_training"]["frozen_question_ids"]) == (
    frozen_ids
)

print(json.dumps(adaptation_hypothesis, indent=2))
print("Adaptation hypothesis validated.")
print("No model request or training operation made.")


{
  "version": 1,
  "decision": "proceed_to_training_data_design",
  "model": "Ministral-3B or another Azure-supported fine-tuning candidate",
  "model_change_policy": "If the fine-tuning model differs from Ministral-3B, establish its unadapted baseline under the identical Milestone 5 runtime contract before training.",
  "fixed_runtime_system": {
    "context_sha256": "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987",
    "instructions_sha256": "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509",
    "output_normalization": "strict single surrounding SQL-fence unwrap",
    "candidate_sql_repair": false
  },
  "hypothesis": "Given complete and correct runtime schema/domain context, the SLM systematically struggles with schema-constrained SQL construction, including multi-table join paths, exact projection, identifier qualification, and multi-step aggregation. SFT/LoRA on verified, leakage-safe, disjoint examples of those behaviors should improve executio

## Save the Milestone 6 decision artifact

The local artifact records the input-run hashes, residual labels, aggregate
evidence, and pre-registered hypothesis. It contains no frozen question-level
content and remains under the ignored `data/runs` directory.


In [12]:
milestone_6_artifact = {
    "artifact_version": 1,
    "milestone": 6,
    "source_run_sha256": {
        "frontier_working": sha256(
            FRONTIER_WORKING_PATH.read_bytes()
        ).hexdigest(),
        "minimal_slm_working": sha256(
            MINIMAL_SLM_WORKING_PATH.read_bytes()
        ).hexdigest(),
        "selected_working": sha256(
            SELECTED_WORKING_PATH.read_bytes()
        ).hexdigest(),
        "official_context_frozen_v2": EXPECTED_FROZEN_V2_SHA256,
    },
    "working_transition_counts": dict(transition_counts),
    "residual_cluster_counts": dict(cluster_counts),
    "residual_labels": {
        str(question_id): label
        for question_id, label in sorted(residual_labels.items())
    },
    "frozen_aggregate_evidence": frozen_aggregate_evidence,
    "adaptation_hypothesis": adaptation_hypothesis,
    "frozen_records_inspected": 0,
}

milestone_6_path = (
    DATA_DIR / "runs" / "adaptation_hypothesis_v1.json"
)
milestone_6_path.parent.mkdir(parents=True, exist_ok=True)
temporary_path = milestone_6_path.with_suffix(".json.tmp")
temporary_path.write_text(
    json.dumps(milestone_6_artifact, indent=2),
    encoding="utf-8",
)
temporary_path.replace(milestone_6_path)

milestone_6_sha256 = sha256(
    milestone_6_path.read_bytes()
).hexdigest()

print("Saved:", milestone_6_path)
print("Milestone 6 artifact SHA-256:", milestone_6_sha256)
print("Frozen records inspected: 0")


Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\adaptation_hypothesis_v1.json
Milestone 6 artifact SHA-256: 6a5c9582c583802a3d5af422d35be284e39a478cd48c1aec0cd42d80ba771a6d
Frozen records inspected: 0


## Milestone 6 checkpoint

The project now has an evidence-backed reason to attempt adaptation, but the
hypothesis remains falsifiable. Milestone 7 must design and manually inspect
training data before any Azure fine-tuning job is launched.

The next notebook should:

1. select only examples disjoint from all 36 evaluation IDs;
2. detect near-duplicate questions and SQL templates across splits;
3. define the exact chat/SFT record format;
4. target the observed behavioral clusters without copying evaluation examples;
5. validate SQL parsing, schema references, and execution;
6. create train/validation splits by semantic template or source group;
7. inspect representative samples and token distributions manually;
8. document FINCH's CC-BY-NC-4.0 research/educational constraint.

No fine-tuning begins until that dataset passes leakage, quality, and provenance
checks.
